In [76]:
# LOAD DATASET
import pandas as pd

data = pd.read_csv("../data/intent_dataset.csv")
data.head()

,text,intent
0,berapa lama andi bekerja,lama_kerja
1,andi sudah kerja berapa tahun,lama_kerja
2,sudah berapa lama andi kerja,lama_kerja
3,lama kerja andi berapa,lama_kerja
4,gaji budi berapa,gaji


In [77]:
# NLP + ML PIPELINE
# Kita pakai TF-IDF + Logistic Regression (ini standar industri).
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

### TRAIN AND EVALUATE MODEL

In [78]:
# TRAIN AND EVALUATE MODEL
X_train, X_test, y_train, y_test = train_test_split(
    data["text"], data["intent"], test_size=0.2, random_state=42
)

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

                       precision    recall  f1-score   support

                 gaji       1.00      1.00      1.00         2
           izin_sakit       0.25      1.00      0.40         1
izin_tanpa_keterangan       0.00      0.00      0.00         3
           lama_kerja       1.00      1.00      1.00         1
            leave_day       0.67      1.00      0.80         2
               posisi       0.00      0.00      0.00         1
       profil_lengkap       0.00      0.00      0.00         1
               status       0.00      0.00      0.00         0

             accuracy                           0.55        11
            macro avg       0.36      0.50      0.40        11
         weighted avg       0.42      0.55      0.45        11



d:\laragon\bin\python\python-3.13\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\laragon\bin\python\python-3.13\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\laragon\bin\python\python-3.13\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\larago

In [79]:
# TEXT NORMALIZATION FUNCTION
import re
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return " ".join(lemmatizer.lemmatize(w) for w in text.split())

In [80]:
# PREDIKSI DENGAN DATA BARU
model.predict([
    "eko udah kerja berapa lama ya",
    "berapa kali eko izin sakitnya",
    "eko masih aktif gak",
    "info cuti eko dong"
])

array(['status', 'izin_sakit', 'status', 'leave_day'], dtype=object)

In [81]:
# PREDIKSI DENGAN DATA BARU
model.predict([
    "berapa lama eko bekerja",
    "gaji agus berapa",
    "izin sakit maya",
    "tampilkan data lengkap siti"
])

array(['lama_kerja', 'gaji', 'izin_sakit', 'profil_lengkap'], dtype=object)

### CROSS VALIDATION

In [84]:
# CROSS VALIDATION
# ini kita masih pake data kecil makannya pake CV = 3 aja karena selebihnya enggak bisa karena data kita kecil
# okey sekarang sudah bisa CV = 5 dan acc nya 83% which is good
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, data["text"], data["intent"], cv=5)
scores.mean()

d:\laragon\bin\python\python-3.13\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


np.float64(0.8654545454545455)

### IMPORT MODEL

In [85]:
# IMPORT MODEL
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)

joblib.dump(model, "../models/intent_model.pkl")

['../models/intent_model.pkl']

---

### Optional enggak wajib di jalanin ya

In [ ]:

# Hasil KFold yang rendah menunjukkan bahwa dataset intent 
# terlalu kecil dan tidak seimbang untuk evaluasi 5-fold yang valid.
from sklearn.model_selection import KFold
scores = cross_val_score(model, data["text"], data["intent"], cv=KFold(n_splits=5))
print(scores)

[0.33333333 0.33333333 0.         0.16666667 0.2       ]


---